In [26]:
import pandas as pd
import numpy as np
from pulp import *

In [27]:
pd.set_option('display.max_columns', None) 
pd.set_option('display.width', 1000)

In [28]:
supply=pd.read_csv("supply_table.csv")
demand=pd.read_csv("demand_table.csv")
lane_mode=pd.read_csv("lane_mode.csv")

In [29]:
supply.columns.tolist()

['Product type', 'Supplier city', 'Supply units']

In [30]:
demand.columns.tolist()

['Product type', 'Distribution Centre', 'Demand units']

In [31]:
lane_mode.columns.tolist()

['Supplier city',
 'Distribution Centre',
 'Transportation modes',
 'Shipping costs',
 'Shipping times']

In [32]:
supsum=supply.groupby('Product type')['Supply units'].sum()
supsum

Product type
cosmetics    1525
haircare     1644
skincare     1608
Name: Supply units, dtype: int64

In [33]:
demsum=demand.groupby('Product type')['Demand units'].sum()
demsum

Product type
cosmetics    1343
haircare     1480
skincare     2099
Name: Demand units, dtype: int64

In [34]:
supply_dict = (
    supply.set_index(['Supplier city', 'Product type'])['Supply units']
    .to_dict()
)

demand_dict = (
    demand.set_index(['Distribution Centre', 'Product type'])['Demand units']
    .to_dict()
)

In [35]:
lane_indexed = lane_mode.set_index(['Supplier city', 'Distribution Centre', 'Transportation modes'])

cost = lane_indexed['Shipping costs']
time = lane_indexed['Shipping times']

COST_WEIGHT = 0.7
TIME_WEIGHT = 0.3

cost_min, cost_max = cost.min(), cost.max()
time_min, time_max = time.min(), time.max()

norm_costs = (cost - cost_min) / (cost_max - cost_min)
norm_times = (time - time_min) / (time_max - time_min)

costs_dict = norm_costs.to_dict()
times_dict = norm_times.to_dict()

In [36]:
Origin = supply['Supplier city'].unique().tolist()
Destination = demand['Distribution Centre'].unique().tolist()
Mode = lane_mode['Transportation modes'].unique().tolist()
Product = supply['Product type'].unique().tolist()

In [37]:
lp=LpProblem("Optimization", LpMinimize)

In [38]:
vars = LpVariable.dicts("quantity",(Origin, Destination, Mode,Product), 0, None, LpInteger)

In [39]:
BIG_M = 5
lp += lpSum(vars[l][r][m][p] * (COST_WEIGHT*costs_dict.get((l, r, m), BIG_M) + TIME_WEIGHT*times_dict.get((l,r,m),BIG_M)) for l in Origin for r in Destination for m in Mode for p in Product)


In [40]:
# ── SUPPLY CONSTRAINTS ─────────────────────────────────────────────────────

for l in Origin:
    for p in Product:
        total_shipped = lpSum(vars[l][r][m][p] for r in Destination for m in Mode)
        supply_limit = supply_dict.get((l, p), 0)
        
        
        if p == "skincare":
            lp += (total_shipped == supply_limit)   # demand > supply: ship everything
        else:
            lp += (total_shipped <= supply_limit)   # demand < supply: cap at available


In [41]:
#--- Demand Constraints----

for r in Destination:
    for p in Product:
        total_ordered = lpSum(vars[l][r][m][p] for l in Origin for m in Mode)
        demand_limit = demand_dict.get((r, p), 0)
        
        if p == "skincare":
            lp+= (total_ordered <= demand_limit)
        else:
            lp+= (total_ordered == demand_limit) 
    

In [42]:
lp.solve()
print("Status:", LpStatus[lp.status])

Status: Optimal


In [43]:

results = [
    {
        "Supplier city": l,
        "Distribution Centre": r,
        "Transportation modes": m,
        "Product": p,
        "Quantity": vars[l][r][m][p].varValue
    }
    for l in Origin for r in Destination for m in Mode for p in Product
    if vars[l][r][m][p].varValue != 0
]


df = pd.DataFrame(results)

print(df.head())
print("Total counts:", len(df))

  Supplier city Distribution Centre Transportation modes    Product  Quantity
0     Bangalore                DC_A                  Sea  cosmetics     226.0
1     Bangalore                DC_A                  Sea   haircare     397.0
2     Bangalore                DC_A                  Sea   skincare     233.0
3       Chennai                DC_A                 Road  cosmetics     150.0
4       Chennai                DC_A                 Road   haircare     225.0
Total counts: 18


In [44]:
final_df = pd.merge(df, lane_mode, on=['Supplier city', 'Distribution Centre', 'Transportation modes'])

final_df['Total_Shipping_Cost'] = final_df['Quantity'] * final_df['Shipping costs']
opt_shipping_cost=final_df['Total_Shipping_Cost'].sum()
print(final_df.head().to_string(index=False))

print("Total Shipping Cost:", opt_shipping_cost)

Supplier city Distribution Centre Transportation modes   Product  Quantity  Shipping costs  Shipping times  Total_Shipping_Cost
    Bangalore                DC_A                  Sea cosmetics     226.0        2.457934             4.0           555.492977
    Bangalore                DC_A                  Sea  haircare     397.0        2.457934             4.0           975.799611
    Bangalore                DC_A                  Sea  skincare     233.0        2.457934             4.0           572.698512
      Chennai                DC_A                 Road cosmetics     150.0        1.512937             5.0           226.940526
      Chennai                DC_A                 Road  haircare     225.0        1.512937             5.0           340.410788
Total Shipping Cost: 12436.804298306246


In [45]:
#historical shipment
sp_d=pd.read_csv("sp_data.csv")

In [46]:
sp_d['Total shipping cost']=sp_d['Quantity']*sp_d['Shipping costs']

In [47]:
grand_total=sp_d['Total shipping cost'].sum()

print('Grand total:',grand_total)

Grand total: 24980.99209618747


In [48]:
model_improvement=((grand_total-opt_shipping_cost)/grand_total)*100
print("Improvement:",model_improvement.round(2),"%")

Improvement: 50.21 %


In [49]:
print('Time improvement:',(((lane_mode['Shipping times'].mean()-final_df['Shipping times'].mean())/lane_mode['Shipping times'].mean())*100).round(2),'%')

Time improvement: 30.91 %


In [50]:
final_df.to_csv("final_df.csv",index=False)

In [51]:
capacity = sp_d.groupby(['Supplier city', 'Distribution Centre', 'Transportation modes'])['Quantity'].sum().reset_index()
capacity = capacity.rename(columns={'Quantity': 'Shipping limit'})
capacity.to_csv('capacity limits.csv', index=False)

In [52]:
capacity_dict = (
    capacity.set_index(['Supplier city', 'Distribution Centre','Transportation modes'])['Shipping limit']
    .to_dict()
)

In [53]:
lp_2=LpProblem("Optimization", LpMinimize)

In [54]:
vars_2 = LpVariable.dicts("quantity",(Origin, Destination, Mode,Product), 0, None,LpInteger)

In [55]:
BIG_M = 5
lp_2 += lpSum(vars_2[l][r][m][p] * (COST_WEIGHT*costs_dict.get((l, r, m), BIG_M) + TIME_WEIGHT*times_dict.get((l,r,m),BIG_M)) for l in Origin for r in Destination for m in Mode for p in Product)


In [56]:
# ──  SUPPLY CONSTRAINTS ─────────────────────────────────────────────────────

for l in Origin:
    for p in Product:
        total_shipped = lpSum(vars_2[l][r][m][p] for r in Destination for m in Mode)
        supply_limit = supply_dict[(l, p)]
        
        
        if p == "skincare":
            lp_2 += (total_shipped == supply_limit)   # demand > supply: ship everything
        else:
            lp_2 += (total_shipped <= supply_limit)   # demand < supply: cap at available
            
#--- Demand Constraints----

for r in Destination:
    for p in Product:
        total_ordered = lpSum(vars_2[l][r][m][p] for l in Origin for m in Mode)
        demand_limit = demand_dict[(r, p)]
        
        if p == "skincare":
            lp_2 += (total_ordered <= demand_limit)
        else:
            lp_2 += (total_ordered == demand_limit) 
    

  

In [57]:
for l in Origin:
    for r in Destination:
        for m in Mode:            
            capacity_limit=capacity_dict.get((l,r,m), 0)          
            lp_2 += lpSum(vars_2[l][r][m][p] for p in Product)<=capacity_limit

In [58]:
lp_2.solve()
print("Status_2:", LpStatus[lp_2.status])

Status_2: Infeasible


In [59]:

results_2 = [
    {
        "Supplier city": l,
        "Distribution Centre": r,
        "Transportation modes": m,
        "Product": p,
        "Quantity": vars_2[l][r][m][p].varValue
    }
    for l in Origin for r in Destination for m in Mode for p in Product
    if vars_2[l][r][m][p].varValue != 0
]


df_2 = pd.DataFrame(results_2)


print(df_2.head())
print("Total counts:", len(df_2))

  Supplier city Distribution Centre Transportation modes    Product  Quantity
0     Bangalore                DC_A                  Air   skincare       4.0
1     Bangalore                DC_A                 Rail   haircare     128.0
2     Bangalore                DC_A                 Road   skincare     173.0
3     Bangalore                DC_A                  Sea   haircare      54.0
4     Bangalore                DC_B                 Rail  cosmetics      87.0
Total counts: 62


In [60]:
final_df2 = pd.merge(df_2, lane_mode, on=['Supplier city', 'Distribution Centre', 'Transportation modes'])

final_df2['Total_Shipping_Cost'] = final_df2['Quantity'] * final_df2['Shipping costs']
opt_shipping_cost_2=final_df2['Total_Shipping_Cost'].sum()
print(final_df2.head())

print("Total Shipping Cost:", opt_shipping_cost_2)

  Supplier city Distribution Centre Transportation modes    Product  Quantity  Shipping costs  Shipping times  Total_Shipping_Cost
0     Bangalore                DC_A                  Air   skincare       4.0        6.966308        4.500000            27.865233
1     Bangalore                DC_A                 Rail   haircare     128.0        4.918780        7.333333           629.603790
2     Bangalore                DC_A                 Road   skincare     173.0        6.186276        4.400000          1070.225666
3     Bangalore                DC_A                  Sea   haircare      54.0        2.457934        4.000000           132.728411
4     Bangalore                DC_B                 Rail  cosmetics      87.0        7.137022        8.000000           620.920952
Total Shipping Cost: 24017.183254302876


In [61]:
model_improvement_2=((grand_total-opt_shipping_cost_2)/grand_total)*100
print("Improvement_2:",model_improvement_2.round(2),"%")

Improvement_2: 3.86 %


In [62]:
final_df2.groupby('Product')['Quantity'].sum()

Product
cosmetics    1343.0
haircare     1480.0
skincare     1608.0
Name: Quantity, dtype: float64

In [63]:
final_df2['Shipping times'].mean()

np.float64(5.710752688172042)

In [64]:
print('Time improvement_2:',(((lane_mode['Shipping times'].mean()-final_df2['Shipping times'].mean())/lane_mode['Shipping times'].mean())*100).round(2),'%')

Time improvement_2: -0.97 %


In [65]:
final_df2.to_csv("final_df2.csv",index=False)

In [66]:
#--------------------------Capacity buffer model---------------------------------------------------------------------------------------

lp_3=LpProblem("Optimization", LpMinimize)

In [67]:
vars_3 = LpVariable.dicts("quantity",(Origin, Destination, Mode,Product), 0, None,LpInteger)

In [68]:
BIG_M = 5
lp_3 += lpSum(vars_3[l][r][m][p] * (COST_WEIGHT*costs_dict.get((l, r, m), BIG_M) + TIME_WEIGHT*times_dict.get((l,r,m),BIG_M)) for l in Origin for r in Destination for m in Mode for p in Product)


In [69]:
# ──  SUPPLY CONSTRAINTS ─────────────────────────────────────────────────────

for l in Origin:
    for p in Product:
        total_shipped = lpSum(vars_3[l][r][m][p] for r in Destination for m in Mode)
        supply_limit = supply_dict[(l, p)]
        
        
        if p == "skincare":
            lp_3 += (total_shipped == supply_limit)   # demand > supply: ship everything
        else:
            lp_3 += (total_shipped <= supply_limit)   # demand < supply: cap at available
            
#--- Demand Constraints----

for r in Destination:
    for p in Product:
        total_ordered = lpSum(vars_3[l][r][m][p] for l in Origin for m in Mode)
        demand_limit = demand_dict[(r, p)]
        
        if p == "skincare":
            lp_3 += (total_ordered <= demand_limit)
        else:
            lp_3 += (total_ordered == demand_limit) 
    

  

In [70]:
for l in Origin:
    for r in Destination:
        for m in Mode:
            capacity_limit=capacity_dict.get((l,r,m), 0)*1.07            
            lp_3 += lpSum(vars_3[l][r][m][p] for p in Product)<=capacity_limit

In [71]:
lp_3.solve()
print("Status_3:", LpStatus[lp_3.status])

Status_3: Optimal


In [72]:
results_3 = [
    {
        "Supplier city": l,
        "Distribution Centre": r,
        "Transportation modes": m,
        "Product": p,
        "Quantity": vars_3[l][r][m][p].varValue
    }
    for l in Origin for r in Destination for m in Mode for p in Product
    if vars_3[l][r][m][p].varValue != 0
]


df_3 = pd.DataFrame(results_3)


print(df_3.head())
print("Total counts:", len(df_3))

  Supplier city Distribution Centre Transportation modes   Product  Quantity
0     Bangalore                DC_A                  Air  skincare       4.0
1     Bangalore                DC_A                 Rail  haircare     136.0
2     Bangalore                DC_A                 Road  haircare      78.0
3     Bangalore                DC_A                 Road  skincare     107.0
4     Bangalore                DC_A                  Sea  skincare      57.0
Total counts: 58


In [73]:
final_df3 = pd.merge(df_3, lane_mode, on=['Supplier city', 'Distribution Centre', 'Transportation modes'])

final_df3['Total_Shipping_Cost'] = final_df3['Quantity'] * final_df3['Shipping costs']
opt_shipping_cost_3=final_df3['Total_Shipping_Cost'].sum()
print(final_df3.head())

print("Total Shipping Cost:", opt_shipping_cost_3)

  Supplier city Distribution Centre Transportation modes   Product  Quantity  Shipping costs  Shipping times  Total_Shipping_Cost
0     Bangalore                DC_A                  Air  skincare       4.0        6.966308        4.500000            27.865233
1     Bangalore                DC_A                 Rail  haircare     136.0        4.918780        7.333333           668.954027
2     Bangalore                DC_A                 Road  haircare      78.0        6.186276        4.400000           482.529491
3     Bangalore                DC_A                 Road  skincare     107.0        6.186276        4.400000           661.931482
4     Bangalore                DC_A                  Sea  skincare      57.0        2.457934        4.000000           140.102211
Total Shipping Cost: 24335.571428296073


In [74]:
model_improvement_3=((grand_total-opt_shipping_cost_3)/grand_total)*100
print("Improvement_3:",model_improvement_3.round(2),"%")

Improvement_3: 2.58 %


In [75]:
final_df3.groupby('Product')['Quantity'].sum()

Product
cosmetics    1343.0
haircare     1480.0
skincare     1608.0
Name: Quantity, dtype: float64

In [76]:
final_df3['Shipping times'].mean()

np.float64(5.50948275862069)

In [77]:
print('Time improvement_3:',(((lane_mode['Shipping times'].mean()-final_df3['Shipping times'].mean())/lane_mode['Shipping times'].mean())*100).round(2),'%')

Time improvement_3: 2.59 %


In [78]:
final_df3.to_csv("final_df3.csv",index=False)